In [36]:
import pandas as pd
df = pd.read_csv(r"MallCustomers.csv")
df.head()
df.describe

<bound method NDFrame.describe of      CustomerID  Gender   Age  Annual Income (k$)  Spending Score (1-100)
0             1    Male  19.0                  15                      39
1             2    Male  21.0                  15                      81
2             3  Female  20.0                  16                       6
3             4  Female  23.0                  16                      77
4             5  Female  31.0                  17                      40
..          ...     ...   ...                 ...                     ...
195         196  Female  35.0                 120                      79
196         197  Female  45.0                 126                      28
197         198    Male   NaN                 126                      74
198         199    Male  32.0                 137                      18
199         200    Male  30.0                 137                      83

[200 rows x 5 columns]>

In [37]:
df['Age_Group'] = pd.cut(
    df['Age'] ,
    bins=[0,25,40,60,100] ,
    labels=['Young','Adult','Senior','Elder']
)

df['Income_Level'] = pd.qcut(
    df['Annual Income (k$)'] , 
    q=3 ,
    labels = ['Low_Income','Medieum_Income','Hight_Income']
)

df['Spending_Level'] = pd.qcut(
    df['Spending Score (1-100)'] , 
    q=3 ,
    labels = ['Low_Spending','Medieum_Spending','Hight_spending']
)

df[['Age_Group','Income_Level','Spending_Level']].head()

,Age_Group,Income_Level,Spending_Level
0,Young,Low_Income,Low_Spending
1,Young,Low_Income,Hight_spending
2,Young,Low_Income,Low_Spending
3,Young,Low_Income,Hight_spending
4,Adult,Low_Income,Low_Spending


In [38]:
transactions = df[['Gender','Age_Group','Income_Level','Spending_Level']]
transactions_list = transactions.astype(str).values.tolist()
transactions_list[:5]

[['Male', 'Young', 'Low_Income', 'Low_Spending'],
 ['Male', 'Young', 'Low_Income', 'Hight_spending'],
 ['Female', 'Young', 'Low_Income', 'Low_Spending'],
 ['Female', 'Young', 'Low_Income', 'Hight_spending'],
 ['Female', 'Adult', 'Low_Income', 'Low_Spending']]

In [39]:
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd
te = TransactionEncoder()
te_array = te.fit(transactions_list).transform(transactions_list)

df_transactions = pd.DataFrame(te_array, columns=te.columns_)
df_transactions.head()

,Adult,Elder,Female,Hight_Income,Hight_spending,Low_Income,Low_Spending,Male,Medieum_Income,Medieum_Spending,Senior,Young,nan
0,False,False,False,False,False,True,True,True,False,False,False,True,False
1,False,False,False,False,True,True,False,True,False,False,False,True,False
2,False,False,True,False,False,True,True,False,False,False,False,True,False
3,False,False,True,False,True,True,False,False,False,False,False,True,False
4,True,False,True,False,False,True,True,False,False,False,False,False,False


In [40]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori (
df_transactions , 
    min_support=0.05 ,
    use_colnames=True
)

frequent_itemsets.sort_values(by='support',ascending=False).head()


,support,itemsets
2,0.56,(Female)
7,0.44,(Male)
0,0.40,(Adult)
5,0.35,(Low_Income)
9,0.34,(Medieum_Spending)


In [41]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets , 
    metric="confidence",
    min_threshold=0.6
)

rules[['antecedents','consequents','support','confidence','lift']].sort_values(by='lift',ascending=False).head()

,antecedents,consequents,support,confidence,lift
47,"(Senior, Hight_Income)","(Female, Low_Spending)",0.055,0.611111,3.492063
27,"(Low_Income, Hight_spending)",(Young),0.075,0.600000,3.157895
44,"(Senior, Female, Hight_Income)",(Low_Spending),0.055,1.000000,2.985075
25,"(Senior, Hight_Income)",(Low_Spending),0.090,1.000000,2.985075
50,"(Senior, Female, Medieum_Income)",(Medieum_Spending),0.065,1.000000,2.941176


In [42]:
interesting_rules = rules[
    (rules['lift'] > 1.2) & 
    (rules['support'] > 0.05)
]

interesting_rules[['antecedents','consequents','support','confidence','lift']]


,antecedents,consequents,support,confidence,lift
0,(Hight_Income),(Adult),0.210,0.636364,1.590909
1,(Hight_spending),(Adult),0.235,0.723077,1.807692
3,(Elder),(Medieum_Spending),0.060,0.750000,2.205882
7,(Medieum_Spending),(Medieum_Income),0.235,0.691176,2.159926
8,(Medieum_Income),(Medieum_Spending),0.235,0.734375,2.159926
9,"(Female, Hight_Income)",(Adult),0.115,0.657143,1.642857
10,"(Female, Hight_spending)",(Adult),0.130,0.722222,1.805556
11,"(Adult, Low_Income)",(Female),0.070,0.700000,1.250000
12,"(Adult, Hight_Income)",(Hight_spending),0.155,0.738095,2.271062
13,"(Adult, Hight_spending)",(Hight_Income),0.155,0.659574,1.998711
